In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
PROJECT_DIR   = Path.cwd().resolve().parent
EXTRACT_DIR   = PROJECT_DIR / 'processed' / 'extraction'
TRANSFORM_DIR = PROJECT_DIR / 'processed' / 'transform'

timeseries_cohort = pd.read_csv(TRANSFORM_DIR / 'hourly_timeseries_60min.csv')
cohort_final      = pd.read_csv(TRANSFORM_DIR / 'cohort_final.csv')
cohort_flow       = pd.read_csv(TRANSFORM_DIR / 'cohort_attrition.csv')

for col in ['intime', 'outtime', 'admittime', 'dischtime', 'deathtime']:
    if col in cohort_final.columns:
        cohort_final[col] = pd.to_datetime(cohort_final[col], errors='coerce')

cohort_stay_ids = set(cohort_final['stay_id'])

print(f"Timeseries: {timeseries_cohort.shape}")
print(f"Cohort stays: {cohort_final['stay_id'].nunique():,}")
display(cohort_flow)

Timeseries: (74230, 69)
Cohort stays: 573


,step,n_subjects,n_hadm,n_stays,timeseries_rows,assessment_rows,stays_removed_from_previous,stays_pct_of_initial
0,0. All ICU stays from extraction,568,850,974,86648,5854,0,100.000000
1,1. Positive ICU LOS,568,850,974,86648,5854,0,100.000000
2,2. ICU LOS >= 24 hours,489,690,775,83689,5588,199,79.568789
3,3. Delirium assessment available after 24 hours,390,514,573,74230,5320,202,58.829569


## EDA: 환자 기본정보

In [4]:
# 범주형 기본정보는 subject 안에서 가장 흔한 값을 대표값으로 사용합니다.
def first_mode(values):
    mode_values = values.dropna().mode()
    if len(mode_values) == 0:
        return np.nan
    return mode_values.iloc[0]

# subject-level 요약 테이블을 만듭니다.
subject_summary = (
    cohort_final
    .sort_values(['subject_id', 'intime'])
    .groupby('subject_id')
    .agg(
        ever_delirium=('ever_delirium', 'max'),
        n_stays=('stay_id', 'nunique'),
        first_age=('anchor_age', 'first'),
        gender=('gender', 'first'),
        race=('race', first_mode),
        admission_type=('admission_type', first_mode),
        first_intime=('intime', 'min'),
        last_outtime=('outtime', 'max'),
        total_icu_los_hours=('icu_los_hours', 'sum'),
    )
    .reset_index()
)

stay_summary = cohort_final[['subject_id', 'stay_id', 'anchor_age', 'gender', 'race', 'admission_type', 'icu_los_hours', 'ever_delirium']].copy()

print('=== Cohort overview ===')
print(f"Subjects: {subject_summary['subject_id'].nunique():,}")
print(f"Stays: {cohort_final['stay_id'].nunique():,}")
print(f"ICU stays per subject: median={subject_summary['n_stays'].median():.1f}, IQR={subject_summary['n_stays'].quantile(0.25):.1f}-{subject_summary['n_stays'].quantile(0.75):.1f}")
print(f"Parkinson cohort period: {cohort_final['intime'].min()} to {cohort_final['outtime'].max()}")

print()
print('=== ever_delirium 분포 (subject-level) ===')
print(subject_summary['ever_delirium'].value_counts().sort_index())
print((subject_summary['ever_delirium'].value_counts(normalize=True).sort_index() * 100).round(2))

print()
print('=== Subject-level numeric summary ===')
display(subject_summary[['first_age', 'n_stays', 'total_icu_los_hours']].describe())

print()
print('=== Subject-level summary by ever_delirium ===')
display(subject_summary.groupby('ever_delirium')[['first_age', 'n_stays', 'total_icu_los_hours']].describe())

print()
print('=== Gender by ever_delirium (%) ===')
display((pd.crosstab(subject_summary['ever_delirium'], subject_summary['gender'], normalize='index') * 100).round(1))

print()
print('=== Admission type by ever_delirium (%) ===')
display((pd.crosstab(stay_summary['ever_delirium'], stay_summary['admission_type'], normalize='index') * 100).round(1))


=== Cohort overview ===
Subjects: 390
Stays: 573
ICU stays per subject: median=1.0, IQR=1.0-1.0
Parkinson cohort period: 2110-12-20 02:37:25 to 2212-10-16 11:44:12

=== ever_delirium 분포 (subject-level) ===
ever_delirium
0    137
1    253
Name: count, dtype: int64
ever_delirium
0    35.13
1    64.87
Name: proportion, dtype: float64

=== Subject-level numeric summary ===


,first_age,n_stays,total_icu_los_hours
count,390.000000,390.000000,390.000000
mean,73.120513,1.469231,198.156521
std,10.603552,1.369077,313.452746
min,24.000000,1.000000,27.247500
25%,67.250000,1.000000,58.555972
50%,74.000000,1.000000,100.638056
75%,80.000000,1.000000,208.284375
max,91.000000,19.000000,4047.868889



=== Subject-level summary by ever_delirium ===


first_age                                                      \
                  count       mean        std   min   25%   50%   75%   max   
ever_delirium                                                                 
0                 137.0  73.481752  10.277833  38.0  69.0  75.0  80.0  91.0   
1                 253.0  72.924901  10.790864  24.0  67.0  74.0  80.0  91.0   

              n_stays            ...            total_icu_los_hours  \
                count      mean  ...  75%   max               count   
ever_delirium                    ...                                  
0               137.0  1.167883  ...  1.0   4.0               137.0   
1               253.0  1.632411  ...  2.0  19.0               253.0   

                                                                         \
                     mean         std        min        25%         50%   
ever_delirium                                                             
0               87.633863   69.933125  27.247500  43.764444   68.261111   
1              258.004758  372.547295  30.466667  74.011111  138.673056   

                                        
                      75%          max  
ever_delirium                           
0              107.894722   513.080000  
1              309.576389  4047.868889  

[2 rows x 24 columns]


=== Gender by ever_delirium (%) ===


gender,F,M
ever_delirium,,
0,30.7,69.3
1,35.2,64.8



=== Admission type by ever_delirium (%) ===


admission_type,DIRECT EMER.,DIRECT OBSERVATION,ELECTIVE,EU OBSERVATION,EW EMER.,OBSERVATION ADMIT,SURGICAL SAME DAY ADMISSION,URGENT
ever_delirium,,,,,,,,
0,3.1,0.0,3.1,0.0,37.5,30.0,8.8,17.5
1,1.7,0.2,1.2,0.2,51.8,27.6,3.4,13.8


## EDA: 섬망 평가 주기

Assessment 횟수, assessment 간격, 첫 assessment까지 걸린 시간, ICU hour/day당 assessment 빈도를 확인합니다.


In [ ]:
# delirium 값이 있는 row만 assessment 시점으로 사용합니다.
assessments = (
    timeseries_cohort
    .loc[timeseries_cohort['delirium'].notna(), ['subject_id', 'stay_id', 'bin', 'hours', 'delirium', 'ever_delirium']]
    .rename(columns={'bin': 'assessment_bin'})
    .sort_values(['stay_id', 'assessment_bin'])
    .reset_index(drop=True)
)

# stay/subject별 assessment count와 assessment 간격을 계산합니다.
assessment_counts_by_stay = assessments.groupby('stay_id').size().rename('assessment_count').reset_index()
assessment_counts_by_subject = assessments.groupby('subject_id').size().rename('assessment_count').reset_index()
assessment_intervals = assessments.groupby('stay_id')['hours'].diff().dropna()
first_assessment_hours = assessments.groupby('stay_id')['hours'].min().rename('first_assessment_hour').reset_index()
assessment_rate = assessment_counts_by_stay.merge(cohort_final[['stay_id', 'icu_los_hours']], on='stay_id', how='left')
assessment_rate['assessments_per_icu_day'] = assessment_rate['assessment_count'] / (assessment_rate['icu_los_hours'] / 24)
assessment_rate['assessments_per_icu_hour'] = assessment_rate['assessment_count'] / assessment_rate['icu_los_hours']

print('=== Delirium assessment rows ===')
print(f"Rows: {len(assessments):,}")
print(f"Stays with assessment: {assessments['stay_id'].nunique():,}")
print(f"Subjects with assessment: {assessments['subject_id'].nunique():,}")
print()
print('Assessment outcome distribution:')
print(assessments['delirium'].value_counts().sort_index())

print()
print('=== Assessment count by stay ===')
display(assessment_counts_by_stay['assessment_count'].describe())

print()
print('=== Assessment count by subject ===')
display(assessment_counts_by_subject['assessment_count'].describe())

print()
print('=== Assessment interval hours within stay ===')
display(assessment_intervals.describe())
print(f"Median/IQR: {assessment_intervals.median():.2f} hours / {assessment_intervals.quantile(0.25):.2f}-{assessment_intervals.quantile(0.75):.2f}")

print()
print('=== First assessment hour within ICU stay ===')
display(first_assessment_hours['first_assessment_hour'].describe())

print()
print('=== Assessment frequency ===')
display(assessment_rate[['assessments_per_icu_hour', 'assessments_per_icu_day']].describe())


## EDA: 검사실(lab) 측정 주기

Lab feature별 측정 횟수, 측정 간격, stay/subject coverage와 전체 lab 측정 간격을 확인합니다.


In [ ]:
# long-format 이벤트에서 lab row만 골라 측정 주기를 계산합니다.
all_events_for_eda = pd.read_csv(OUTPUT_DIR / 'all_events_filtered.csv')
all_events_for_eda['charttime'] = pd.to_datetime(all_events_for_eda['charttime'], errors='coerce')
lab_events = all_events_for_eda[
    all_events_for_eda['source_table'].eq('labevents')
    & all_events_for_eda['stay_id'].isin(cohort_stay_ids)
    & all_events_for_eda['charttime'].notna()
].copy()

lab_events = lab_events.merge(
    cohort_final[['stay_id', 'subject_id', 'intime', 'outtime']],
    on=['stay_id', 'subject_id'],
    how='inner',
)
lab_events['lab_hour'] = (lab_events['charttime'] - lab_events['intime']).dt.total_seconds() / 3600
lab_events = lab_events[(lab_events['lab_hour'] >= 0) & (lab_events['charttime'] <= lab_events['outtime'])].copy()

# feature별 측정 횟수와 coverage를 요약합니다.
lab_feature_counts = (
    lab_events
    .groupby('feature_name')
    .agg(
        n_measurements=('feature_name', 'size'),
        n_stays=('stay_id', 'nunique'),
        n_subjects=('subject_id', 'nunique'),
    )
    .reset_index()
)
lab_feature_counts['stay_coverage_pct'] = lab_feature_counts['n_stays'] / cohort_final['stay_id'].nunique() * 100
lab_feature_counts['subject_coverage_pct'] = lab_feature_counts['n_subjects'] / cohort_final['subject_id'].nunique() * 100

# 같은 stay 안에서 같은 lab feature가 반복 측정된 간격을 계산합니다.
lab_sorted = lab_events.sort_values(['stay_id', 'feature_name', 'charttime'])
lab_sorted['interval_hours'] = lab_sorted.groupby(['stay_id', 'feature_name'])['charttime'].diff().dt.total_seconds() / 3600
lab_interval_summary = (
    lab_sorted
    .dropna(subset=['interval_hours'])
    .groupby('feature_name')['interval_hours']
    .agg(
        interval_n='count',
        interval_median_hours='median',
        interval_q1_hours=lambda x: x.quantile(0.25),
        interval_q3_hours=lambda x: x.quantile(0.75),
    )
    .reset_index()
)

lab_summary = (
    lab_feature_counts
    .merge(lab_interval_summary, on='feature_name', how='left')
    .sort_values(['n_measurements', 'stay_coverage_pct'], ascending=False)
    .reset_index(drop=True)
)

# 어떤 lab이든 측정된 시점 기준의 전체 lab 간격도 함께 봅니다.
any_lab_times = lab_events[['stay_id', 'charttime']].drop_duplicates().sort_values(['stay_id', 'charttime']).copy()
any_lab_intervals = any_lab_times.groupby('stay_id')['charttime'].diff().dt.total_seconds().div(3600).dropna()

print('=== Lab measurement overview ===')
print(f"Lab rows: {len(lab_events):,}")
print(f"Lab features: {lab_events['feature_name'].nunique():,}")
print(f"Stays with any lab: {lab_events['stay_id'].nunique():,} / {cohort_final['stay_id'].nunique():,}")
print(f"Subjects with any lab: {lab_events['subject_id'].nunique():,} / {cohort_final['subject_id'].nunique():,}")

print()
print('=== Lab feature measurement summary ===')
display(lab_summary)

print()
print('=== Any-lab interval hours within stay ===')
display(any_lab_intervals.describe())
print(f"Median/IQR: {any_lab_intervals.median():.2f} hours / {any_lab_intervals.quantile(0.25):.2f}-{any_lab_intervals.quantile(0.75):.2f}")
